# RescueBite AI Model C — Donor Surplus Prediction: Data Audit Notebook

> **Synthetic Data Disclaimer**: This dataset is synthetically generated for demonstration and simulation purposes. All metrics, donor records, and surplus quantities reflect simulation experiments and do not represent live operational guarantees.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Robust repository path resolution
current_dir = Path.cwd()
repo_root = current_dir if (current_dir / 'dataset').exists() else current_dir.parent

donations_path = repo_root / 'dataset' / 'donations.csv'
donors_path = repo_root / 'dataset' / 'donors.csv'
holidays_path = repo_root / 'dataset' / 'holidays.csv'
allocations_path = repo_root / 'dataset' / 'allocations.csv'

df_donations = pd.read_csv(donations_path)
df_donors = pd.read_csv(donors_path)
df_holidays = pd.read_csv(holidays_path)
df_allocations = pd.read_csv(allocations_path)

print('Dataset files loaded successfully:')
print(f'  - Donations: {df_donations.shape}')
print(f'  - Donors: {df_donors.shape}')
print(f'  - Holidays: {df_holidays.shape}')
print(f'  - Allocations (context): {df_allocations.shape}')

## 1. Schema, Columns & Data Types

In [ ]:
print('--- DONATIONS SCHEMA ---')
print(df_donations.dtypes)

print('\n--- DONORS SCHEMA ---')
print(df_donors.dtypes)

print('\n--- HOLIDAYS SCHEMA ---')
print(df_holidays.dtypes)

## 2. Integrity Checks: Missing Values & Duplicate Primary Keys

In [ ]:
print('Donations missing values:\n', df_donations.isnull().sum())
print('\nDonors missing values:\n', df_donors.isnull().sum())
print('\nHolidays missing values:\n', df_holidays.isnull().sum())

print('\nDuplicate donation_ids:', df_donations['donation_id'].duplicated().sum())
print('Duplicate donor_ids in donors.csv:', df_donors['donor_id'].duplicated().sum())

## 3. Date Parsing & Temporal Span

In [ ]:
df_donations['created_dt'] = pd.to_datetime(df_donations['created_at'])
df_donations['calendar_date'] = df_donations['created_dt'].dt.date

min_date = df_donations['calendar_date'].min()
max_date = df_donations['calendar_date'].max()
date_span = (max_date - min_date).days + 1

print(f'Temporal Range: {min_date} to {max_date} ({date_span} total calendar days)')
print(f'Unique Active Calendar Dates: {df_donations["calendar_date"].nunique()}')

## 4. Quantity Distribution & Target Statistics

In [ ]:
print('--- Individual Donation Quantity (quantity_meals) ---')
print(df_donations['quantity_meals'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

# Aggregate to Daily Donor Surplus
df_donor_day = df_donations.groupby(['donor_id', 'calendar_date'])['quantity_meals'].agg(
    donation_count='count',
    total_surplus_meals='sum'
).reset_index()

print('\n--- Daily Aggregated Donor Surplus (total_surplus_meals) ---')
print(df_donor_day['total_surplus_meals'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

## 5. Donor-Day Observation Window & Zero-Surplus Analysis

In [ ]:
all_donors = df_donors['donor_id'].unique()
all_dates = pd.date_range(min_date, max_date, freq='D').date
total_grid_size = len(all_donors) * len(all_dates)
active_donor_days = len(df_donor_day)

print(f'Total Registered Donors: {len(all_donors)}')
print(f'Full Cartesian Grid Size (120 donors x 540 dates): {total_grid_size} donor-days')
print(f'Observed Active Donation Days: {active_donor_days} ({active_donor_days / total_grid_size:.2%})')
print(f'Unrecorded (Zero Surplus) Days: {total_grid_size - active_donor_days} ({(total_grid_size - active_donor_days) / total_grid_size:.2%})')

print('\nDonation frequency per active donor-day:')
print(df_donor_day['donation_count'].value_counts())

## 6. Donor Categories & Surplus Profiles

In [ ]:
merged = df_donations.merge(df_donors, on='donor_id')
print('Donations and Average Surplus by Donor Type:')
print(merged.groupby('donor_type')['quantity_meals'].agg(['count', 'mean', 'median', 'std']))